# HL-CUP25 Heterogeneous Ensemble

In [8]:
# --- IMPORTS ---
import os
# Set Keras Backend to PyTorch BEFORE importing anything that might use Keras
os.environ["KERAS_BACKEND"] = "torch"

import time
import json
import torch
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import keras

# Custom Modules
from utils.data_loader import get_ml_cup_data
from models import StandardFeedForwardNet, ModelWithHead, ReadoutAdapter
from utils.training_utils import evaluate, evaluate_mee
import utils.training_utils as training_utils
from executors.executors import build_model_single_hidden, build_model_two_hidden

print("Imports successful. Keras Backend set to torch.")


Imports successful. Keras Backend set to torch.


In [20]:
# --- CONFIGURATION ---
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Ensemble Settings
ENSEMBLE_SIZE_PYTORCH = 10
ENSEMBLE_SIZE_KERAS = 0
ENSEMBLE_SIZE_SVM = 5

# Results CSV Pointers (Link your specific result files here)
# Set to None to try finding them automatically
PYTORCH_RESULTS_CSV = "pytorch/models/optuna/20_01_2026(2layer)/optuna_results.csv"
KERAS_RESULTS_CSV = None # e.g. "keras/models/optuna/.../hp.json" (Using JSON for Keras as logic expects dict)
SVM_RESULTS_DIR = "svm/models/FineTuned/SVR_1"  # Folder containing svr_target_*_trials.csv

USE_GAUSSIAN_NOISE = True # Enforcing diversity via noise for retraining
NOISE_STD = 0.1


# Early Stopping
EARLY_STOPPING_PATIENCE = 250
print(f"Device: {DEVICE}")


Device: mps


In [10]:
# --- CUSTOM CLASS: MULTI-OUTPUT SVR ---
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.svm import SVR

class MultiOutputSVR(BaseEstimator, RegressorMixin):
    """
    A wrapper that holds 4 independent SVR models (one per target).
    This allows us to treat the 4-target regressor as a single 'Model' in the ensemble list.
    """
    def __init__(self, estimators_list=None):
        # estimators_list: list of 4 SVR instances (configured with their specific params)
        self.estimators_list = estimators_list
        self.fitted_estimators_ = []

    def fit(self, X, y):
        # X: (N, F), y: (N, 4)
        self.fitted_estimators_ = []
        n_targets = y.shape[1]
        
        if len(self.estimators_list) != n_targets:
            raise ValueError(f"Expected {n_targets} estimators, got {len(self.estimators_list)}")
            
        for i in range(n_targets):
            est = clone(self.estimators_list[i])
            est.fit(X, y[:, i])
            self.fitted_estimators_.append(est)
            
        return self

    def predict(self, X):
        preds = []
        for est in self.fitted_estimators_:
            preds.append(est.predict(X))
        return np.column_stack(preds)


In [11]:
# --- ENSEMBLE CLASS ---
class HeterogeneousEnsemble:
    def __init__(self, pytorch_models, keras_models, svm_models, target_scaler=None):
        self.pytorch_models = pytorch_models
        self.keras_models = keras_models
        self.svm_models = svm_models 
        self.target_scaler = target_scaler
        
    def eval(self):
        """Mock eval method to satisfy evaluate_mee"""
        pass

    def __call__(self, x):
        """Allows the ensemble to be called like a tensor function: model(x)"""
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x
            
        # Get predictions (SCALED)
        pred_np = self.predict(x_np)
        
        # Return as tensor
        return torch.tensor(pred_np, dtype=torch.float32, device=DEVICE)

    def predict(self, X_numpy):
        # Accumulate predictions in SCALED space
        preds = []
        
        # 1. PyTorch
        if self.pytorch_models:
            X_torch = torch.tensor(X_numpy, dtype=torch.float32).to(DEVICE)
            for i, m in enumerate(self.pytorch_models):
                m.eval()
                with torch.no_grad():
                    out = m(X_torch).cpu().numpy()
                    # print(f"DEBUG: PyTorch Model {i} Mean={out.mean():.4f} Std={out.std():.4f}")
                    preds.append(out)
                    
        # 2. Keras
        if self.keras_models:
            for i, m in enumerate(self.keras_models):
                out = m.predict(X_numpy, verbose=0)
                # print(f"DEBUG: Keras Model {i} Mean={out.mean():.4f} Std={out.std():.4f}")
                preds.append(out)
        
        # 3. SVM
        if self.svm_models:
            for i, m in enumerate(self.svm_models):
                out = m.predict(X_numpy)
                # print(f"DEBUG: SVM Model {i} RAW Mean={out.mean():.4f} Std={out.std():.4f}")
                
                if self.target_scaler:
                    if out.ndim == 1:
                        out = out.reshape(-1, 1)
                    # print(f"DEBUG: Target Transform Mean={self.target_scaler.mean_} Scale={self.target_scaler.scale_}")     
                    out = self.target_scaler.transform(out)
                    # print(f"DEBUG: SVM Model {i} SCALED Mean={out.mean():.4f} Std={out.std():.4f}")
                    
                preds.append(out)

        if not preds:
            return np.zeros((X_numpy.shape[0], OUTPUT_SIZE))
            
        # Check consistency
        preds_np = np.array(preds)
        # print(f"DEBUG: Ensemble Preds Shape {preds_np.shape}. Mean of means: {np.mean(preds_np, axis=0).mean():.4f}")
        
        # Return MEAN of scaled predictions
        return np.mean(preds, axis=0)


In [13]:
# --- EARLY STOPPING CLASS ---
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.mode = mode
        self.best_score = None
        self.early_stop = False
        self.best_state = None

    def __call__(self, score, model=None):
        if self.best_score is None:
            self.best_score = score
            if model: self.best_state = model.state_dict()
        elif self._is_better(score, self.best_score):
            self.best_score = score
            self.counter = 0
            if model: self.best_state = model.state_dict()
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def _is_better(self, current, best):
        if self.mode == 'min':
            return current < (best - self.min_delta)
        else:
            return current > (best + self.min_delta)

## Part 1: Ensemble selection 

In [ ]:
# --- DATA LOADING ---
# Load ML-CUP Data
train_loader, val_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE, target_scaler = get_ml_cup_data(
    BATCH_SIZE, 
    validation_ratio=0.20,
    test_ratio=0.25,
    scale_target=True,
    num_workers=0
)

print(f"Data Loaded. Input: {INPUT_SIZE}, Output: {OUTPUT_SIZE}")

# --- PREPARE K-FOLD DATA ---
from sklearn.model_selection import KFold
from torch.utils.data import TensorDataset, DataLoader

# Merge Train and Val for K-Fold
# Note: get_ml_cup_data returns Loaders. We access underlying dataset tensors.
X_train_main = train_loader.dataset.X
y_train_main = train_loader.dataset.y
X_val_main = val_loader.dataset.X
y_val_main = val_loader.dataset.y

X_dev = torch.cat([X_train_main, X_val_main], dim=0)
y_dev = torch.cat([y_train_main, y_val_main], dim=0)

print(f"Merged Development Set: {X_dev.shape}")

# Initialize K-Fold
# n_splits matches ENSEMBLE_SIZE_PYTORCH so each model gets a different fold config.
if ENSEMBLE_SIZE_PYTORCH > 0:
    kf = KFold(n_splits=ENSEMBLE_SIZE_PYTORCH, shuffle=True, random_state=42)
    folds = list(kf.split(X_dev))
    print(f"Created {len(folds)} folds for PyTorch Ensemble.")


In [ ]:
# --- HELPER: IMPORT HYPERPARAMETERS FROM STUDIES ---
import optuna
import glob
import json
import os
import pandas as pd
import ast

# --- CONFIGURATION: PARAMETER LIMIT CHECK ---
LIMIT_PARAMS = False   # Set to True to activate the filter
LIMIT_COUNT = 900
INPUT_DIM = 12            # REPLACE with your dataset's output size (number of targets)
OUTPUT_DIM = 4            # REPLACE with your dataset's output size (number of targets)

def get_model_params(row):
    """
    Calculates total parameters (Weights + Biases) for an MLP.
    Formula: 
      1. Input -> Hidden 1: (In * Hid) + Hid
      2. Hidden -> Hidden:  (Hid * Hid) + Hid  (for n_layers - 1)
      3. Hidden -> Output:  (Hid * Out) + Out
    """
    n_layers = int(row.get('params_n_layers', 2)) # Default to 2 if missing
    hidden_size = int(row['params_hidden_size'])
    
    # Layer 1: Input -> Hidden
    total_params = (INPUT_DIM * hidden_size) + hidden_size
    
    # Middle Layers: Hidden -> Hidden
    if n_layers > 1:
        total_params += (n_layers - 1) * ((hidden_size * hidden_size) + hidden_size)
        
    # Output Layer: Hidden -> Output
    total_params += (hidden_size * OUTPUT_DIM) + OUTPUT_DIM
    
    return total_params

def load_best_params():
    params = {}
    
    # 1. PYTORCH
    try:
        if PYTORCH_RESULTS_CSV and os.path.exists(PYTORCH_RESULTS_CSV):
            df = pd.read_csv(PYTORCH_RESULTS_CSV)

            # --- CHECK: Filter for model complexity ---
            if LIMIT_PARAMS:
                print(f"[PyTorch] Filtering for models with < {LIMIT_COUNT} parameters...")
                
                # Calculate parameters for every row
                df['calculated_params'] = df.apply(get_model_params, axis=1)
                
                # Filter the dataframe
                df_constrained = df[df['calculated_params'] < LIMIT_COUNT]
                
                if not df_constrained.empty:
                    print(f"  > Found {len(df_constrained)} trials matching criteria.")
                    df = df_constrained
                else:
                    print(f"  > WARNING: No trials found with < {LIMIT_COUNT} params. Reverting to all trials.")
            # ------------------------------------------

            best_idx = df['value'].idxmin()
            best_row = df.loc[best_idx]
            p_dict = {k.replace('params_', ''): v for k, v in best_row.items() if k.startswith('params_')}
            params['pytorch'] = p_dict
            
            # Log info about selected model
            sel_val = best_row['value']
            # Use calculated params if available, otherwise calculate it now for display
            sel_params = best_row['calculated_params'] if 'calculated_params' in best_row else get_model_params(best_row)
            
            print(f"Loaded PyTorch Best Params (Loss: {sel_val:.4f}, Size: {sel_params}): {params['pytorch']}")
            
        else:
             # Fallback for SQL (Cannot easily filter by calculated params without loading all)
             storage = "sqlite:///optuna_mlcup_nn.db"
             if os.path.exists("optuna_mlcup_nn.db"):
                 study = optuna.load_study(study_name="mlcup_search", storage=storage)
                 params['pytorch'] = study.best_params
                 print(f"Loaded PyTorch Best Params from DB: {params['pytorch']}")
             else:
                 params['pytorch'] = None
    except Exception as e:
        print(f"Could not load PyTorch params: {e}")
        params['pytorch'] = None

    # 2. KERAS (Unchanged logic)
    if KERAS_RESULTS_CSV and os.path.exists(KERAS_RESULTS_CSV):
        try:
            with open(KERAS_RESULTS_CSV, 'r') as f:
                params['keras'] = json.load(f)
            print(f"Loaded Keras Best Params: {params['keras']}")
        except: params['keras'] = None
    else:
        keras_files = glob.glob("keras/models/optuna/**/hp.json", recursive=True)
        if keras_files:
            latest_file = max(keras_files, key=os.path.getmtime)
            with open(latest_file, 'r') as f:
                params['keras'] = json.load(f)
            print(f"Loaded Keras Best Params from {latest_file}: {params['keras']}")
        else:
            params['keras'] = None

    # 3. SVM (Unchanged logic)
    print("Scanning for SVM results...")
    svm_base_dir = "svm/models"
    all_csvs = glob.glob(os.path.join(svm_base_dir, "**", "*trials.csv"), recursive=True)
    
    best_svm = {}
    if all_csvs:
        for t_idx in range(4): # 4 targets
            target_str = f"target_{t_idx}"
            target_dfs = []
            
            for fpath in all_csvs:
                 if target_str in os.path.basename(fpath):
                     try:
                         df_t = pd.read_csv(fpath)
                         target_dfs.append(df_t)
                     except: pass
            
            if target_dfs:
                full_df = pd.concat(target_dfs, ignore_index=True)
                full_df = full_df.sort_values(by='value', ascending=False)
                best_row = full_df.iloc[0]
                p_dict = {k.replace('params_', ''): v for k, v in best_row.items() if k.startswith('params_')}
                best_svm[target_str] = p_dict
                print(f"  {target_str}: Best Value={best_row['value']:.4f} Params={p_dict}")
            else:
                best_svm[target_str] = None
        params['svm'] = best_svm
    else:
         print("No SVM CSVs found.")
         params['svm'] = None
         
    return params

START_PARAMS = load_best_params()

In [ ]:
# --- ROBUST INITIALIZATION ---
# Initialize model lists to empty to avoid NameError if specific sections are skipped.
pytorch_models = []
keras_models = []
svm_models = []
print("Model lists initialized to empty.")


In [ ]:
# --- PART 1: PYTORCH MODELS ---
pytorch_models = []
pytorch_histories = {'train_mee': [], 'val_mee': []}

if START_PARAMS['pytorch']:
    print(f"Using Loaded PyTorch Params: {START_PARAMS['pytorch']}")
    pt_params = START_PARAMS['pytorch'].copy()
else:
    print("Using Default PyTorch Params.")
    # Added default dropout here
    pt_params = {'hidden_size': 32, 'lr': 0.01, 'momentum': 0.9, 'activation': 'relu', 'dropout': 0.0}

def train_pytorch_model_with_tracking(idx):
    print(f"Training PyTorch Model {idx+1}/{ENSEMBLE_SIZE_PYTORCH} (Fold {idx+1})...")
    
    # 1. Get Data for this Fold
    train_idx, val_idx = folds[idx]
    X_f_train, y_f_train = X_dev[train_idx], y_dev[train_idx]
    X_f_val, y_f_val = X_dev[val_idx], y_dev[val_idx]
    
    # Create Loader for shuffling
    f_dataset = TensorDataset(X_f_train, y_f_train)
    f_loader = DataLoader(f_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # 2. Build Model
    # Get the number of layers (default to 2 if not found)
    if pt_params.get('n_layers') is None: 
        n_layers = 2
    else:
        n_layers = int(pt_params.get('n_layers', 2)) 
    # Get the size of the layers
    layer_size = int(pt_params.get('hidden_size', 32))
    # Create the list (e.g., if n_layers=3, hidden_sizes=[32, 32, 32])
    hidden_sizes = [layer_size] * n_layers
    act = pt_params.get('activation', 'relu')
    dropout_val = float(pt_params.get('dropout', 0.0))
    
    # Instantiate Base with Dropout
    # Assuming StandardFeedForwardNet accepts 'dropout' as an argument
    base = StandardFeedForwardNet(INPUT_SIZE, hidden_sizes, OUTPUT_SIZE, act, dropout=dropout_val)
    model = ModelWithHead(base, ReadoutAdapter(OUTPUT_SIZE, OUTPUT_SIZE, 'regression')).to(DEVICE)
    
    lr = float(pt_params.get('lr', 0.01))
    mom = float(pt_params.get('momentum', 0.9))
    
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=mom, nesterov=True)
    crit = torch.nn.MSELoss()
    
    # Trackers
    fold_train_mee = []
    fold_val_mee = []
    
    # 3. Training Loop
    epochs = int(pt_params.get('epochs', 1000))
    print(f"Training for {epochs} epochs...")
    early_stopper = EarlyStopper(patience=EARLY_STOPPING_PATIENCE, mode='min')
    for epoch in range(epochs):
        model.train() # Enable Dropout
        for x, y in f_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if USE_GAUSSIAN_NOISE:
                noise = torch.randn_like(x) * NOISE_STD
                x = x + noise
            opt.zero_grad()
            out = model(x)
            loss = crit(out, y)
            loss.backward()
            opt.step()
        
        # --- MEE EVALUATION (Every Epoch) ---
        # Evaluate on full fold arrays (Unscaled)
        model.eval() # Disable Dropout for Evaluation
        with torch.no_grad():
            # Prepare Data
            x_tr_d = X_f_train.to(DEVICE)
            x_val_d = X_f_val.to(DEVICE)
            
            # Predict
            pred_tr = model(x_tr_d).cpu().numpy()
            pred_val = model(x_val_d).cpu().numpy()
            
            # Unscale Targets & Predictions
            y_tr_real = target_scaler.inverse_transform(y_f_train.numpy())
            pred_tr_real = target_scaler.inverse_transform(pred_tr)
            
            y_val_real = target_scaler.inverse_transform(y_f_val.numpy())
            pred_val_real = target_scaler.inverse_transform(pred_val)
            
            # Calculate Mean Euclidean Error
            mee_tr = np.mean(np.linalg.norm(y_tr_real - pred_tr_real, axis=1))
            mee_val = np.mean(np.linalg.norm(y_val_real - pred_val_real, axis=1))
            
            fold_train_mee.append(mee_tr)
            fold_val_mee.append(mee_val)
            
            # EARLY STOPPING CHECK
            early_stopper(mee_val, model)
            if early_stopper.early_stop:
                print(f"  Early stopping at epoch {epoch} (Best MEE: {early_stopper.best_score:.4f})")
                break
            
    
    # Load Best Weights if check triggered, otherwise keep last
    if early_stopper.best_state:
        model.load_state_dict(early_stopper.best_state)
            
    return model, fold_train_mee, fold_val_mee

start_time = time.time()
actual_runs = min(ENSEMBLE_SIZE_PYTORCH, len(folds))

for i in range(actual_runs):
    model, tr_hist, val_hist = train_pytorch_model_with_tracking(i)
    pytorch_models.append(model)
    pytorch_histories['train_mee'].append(tr_hist)
    pytorch_histories['val_mee'].append(val_hist)

print(f"PyTorch Ensemble Training Time: {time.time() - start_time:.2f}s")

In [ ]:
# --- PLOT TRAINING MEE ---
plt.figure(figsize=(14, 6))

# Subplot 1: Training MEE
plt.subplot(1, 2, 1)
for i, curve in enumerate(pytorch_histories['train_mee']):
    plt.plot(curve, label=f'Model {i+1} (Fold {i+1})', alpha=0.6)
plt.title("PyTorch Ensemble: Training MEE (Unscaled)")
plt.xlabel("Epoch")
plt.ylabel("MEE")
plt.yscale('log')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.legend()

# Subplot 2: Validation MEE
plt.subplot(1, 2, 2)
for i, curve in enumerate(pytorch_histories['val_mee']):
    plt.plot(curve, label=f'Model {i+1} (Fold {i+1})', alpha=0.6)
plt.title("PyTorch Ensemble: Validation MEE (Unscaled)")
plt.xlabel("Epoch")
plt.ylabel("MEE")
plt.yscale('log')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- PART 2: KERAS MODELS ---
keras_models = []

if START_PARAMS['keras']:
    print(f"Using Loaded Keras Params: {START_PARAMS['keras']}")
    k_params = START_PARAMS['keras']
    # Config for Keras Executor builders
    # Maps JSON params to Builder params
    # Expected keys in k_params (from hp.json): learning_rate, lambda_1, activation_1, etc.
    
    # Prepare Metadata for builder
    # We need to know input shape. INPUT_SIZE is from loader.
    # Note: Loader input_size might be wrong if 'apply_pca' logic not mirrored here? 
    # Assuming loader handles features.
    k_meta = {"n_features_in_": INPUT_SIZE, "n_outputs_": OUTPUT_SIZE}
    
    def train_keras_model(idx):
        print(f"Training Keras Model {idx+1}/{ENSEMBLE_SIZE_KERAS}...")
        # Seed Logic: varies per model for diversity
        seed = 42 + idx
        
        # Extract params
        lr = k_params.get('learning_rate', 0.01)
        l1 = k_params.get('lambda_1', 1e-4)
        act1 = k_params.get('activation_1', 'relu')
        drop1 = k_params.get('dropout_1', 0.2)
        
        l2 = k_params.get('lambda_2', None)
        if l2: 
            # Two Hidden Layers
            # Need to find unit1/unit2. Usually optuna json has 'unit1'? Or we infer?
            # Looking at executors, 'unit1' is passed. 
            # If not in params, default to 12
            u1 = k_params.get('unit1', 12)
            u2 = k_params.get('unit2', 12)
            act2 = k_params.get('activation_2', 'relu')
            drop2 = k_params.get('dropout_2', 0.2)
            
            model = build_model_two_hidden(
                k_meta, u1, u2, seed, lr, drop1, drop2, l1, l2, act1, act2
            )
        else:
            # Single Hidden
            u1 = k_params.get('unit1', 12)
            model = build_model_single_hidden(
                k_meta, u1, seed, lr, drop1, l1, act1
            )
            
        # Training
        # We need numpy data
        X_tr = train_loader.dataset.X.numpy()
        y_tr = train_loader.dataset.y.numpy()
        if target_scaler:
             # Loader scales X? Yes.
             # Y? If scale_target=True, Y is scaled.
             pass
             
        if USE_GAUSSIAN_NOISE:
             noise = np.random.normal(0, NOISE_STD, X_tr.shape)
             X_tr = X_tr + noise
             
        # Fit
        model.fit(X_tr, y_tr, epochs=500, verbose=0, batch_size=BATCH_SIZE)
        return model

    for i in range(ENSEMBLE_SIZE_KERAS):
         try:
             keras_models.append(train_keras_model(i))
         except Exception as e:
             print(f"Keras Train Error: {e}")
             
else:
    print("No Keras Params loaded. Skipping Keras.")


In [ ]:
# --- PART 3: SVM MODELS ---
from sklearn.ensemble import BaggingRegressor

svm_models = [] # Will contain a single MultiOutputSVR that wraps 4 BaggingRegressors

if START_PARAMS['svm']:
    print(f"Training SVM Ensemble with Bagging (n_estimators={ENSEMBLE_SIZE_SVM})...")
    svm_params_dict = START_PARAMS['svm']
    
    X_tr = train_loader.dataset.X.numpy()
    y_tr = train_loader.dataset.y.numpy()
    
    # --- CRITICAL FIX: Inverse Scale Targets for SVM ---
    # SVM hyperparameters were optimized on UNSCALED targets.
    # We must ensure we train on the same distribution.
    if target_scaler:
        print("Inverse scaling targets for SVM training...")
        y_tr = target_scaler.inverse_transform(y_tr)
    # ---------------------------------------------------
    
    estimators_list = []
    valid_svm = True
    
    for t_idx in range(OUTPUT_SIZE):
        target_key = f"target_{t_idx}"
        conf = svm_params_dict.get(target_key)
        
        if not conf:
            print(f"Missing params for {target_key}, skipping SVM.")
            valid_svm = False
            break
            
        # 1. Base SVR
        base_svr = SVR(kernel='rbf', C=conf.get('C', 1.0), epsilon=conf.get('epsilon', 0.1), gamma=conf.get('gamma', 'scale'))
        
        # 2. Wrap in BaggingRegressor
        # exact logic from svm_analysis.ipynb usually
        regr = BaggingRegressor(
            estimator=base_svr,
            n_estimators=ENSEMBLE_SIZE_SVM, 
            random_state=42 + t_idx,
            max_samples=0.8, # Subsample for diversity
            bootstrap=True,
            n_jobs=-1
        )
        estimators_list.append(regr)
    
    if valid_svm:
        # Wrap 4 BaggingRegressors into one Multi-Output Model
        print("Fitting Bagged SVMs...")
        msvr = MultiOutputSVR(estimators_list=estimators_list)
        msvr.fit(X_tr, y_tr)
        
        svm_models.append(msvr)
        print("SVM Training Complete.")
    
else:
    print("No SVM params loaded. Skipping.")


In [ ]:
# --- INTERNAL EVALUATION ---
full_ensemble = HeterogeneousEnsemble(pytorch_models, keras_models, svm_models, target_scaler=target_scaler)

X_test_np = test_loader.dataset.X.numpy()
y_test_np = test_loader.dataset.y.numpy()

print("Evaluating Ensemble on Internal Test Set...")

# Calculate MEE
# IMPORTANT: Pass target_scaler to unscale both predictions and targets
mee = evaluate_mee(full_ensemble, test_loader, DEVICE, target_scaler=target_scaler)
print(f"Ensemble Internal Test MEE: {mee:.4f}")

# --- SAVE BEST ENSEMBLE LOGIC ---
best_score_file = "ensemble_best_score.json"
current_best = float('inf')
if os.path.exists(best_score_file):
    with open(best_score_file, 'r') as f:
        try:
            data = json.load(f)
            current_best = data.get('best_mee', float('inf'))
        except: pass

print(f"Previous Best MEE: {current_best:.4f}")

if mee < current_best:
    print("New Best Ensemble found! Saving configuration to JSON...")
    
    # Save Configuration ONLY (No weights)
    ensemble_config = {
        'best_mee': float(mee),
        'timestamp': time.strftime("%Y-%m-%d %H:%M:%S"),
        'config': {
            'pytorch': {
                'count': ENSEMBLE_SIZE_PYTORCH,
                'params': START_PARAMS['pytorch'],
                'epochs': int(pt_params.get('epochs', 1200)) if 'pt_params' in globals() else 1200
            },
            'keras': {
                'count': ENSEMBLE_SIZE_KERAS,
                'params': START_PARAMS['keras']
            },
            'svm': {
                'count': ENSEMBLE_SIZE_SVM,
                'params': START_PARAMS['svm']
            }
        }
    }
    
    # Save best parameters to JSON
    with open("best_ensemble_config.json", 'w') as f:
        json.dump(ensemble_config, f, indent=4, default=lambda o: o.item())

        
    # Update score file for tracking
    with open(best_score_file, 'w') as f:
        json.dump({'best_mee': float(mee)}, f)
        
    print("Configuration saved to best_ensemble_config.json")
else:
    print("Ensemble did not improve over best recorded score.")

## Part 2: blind test set

In [ ]:
# --- PART 2: BLIND TEST SET (RELOAD & RETRAIN) ---
import os
import json
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from torch.utils.data import TensorDataset, DataLoader
from models import StandardFeedForwardNet, ModelWithHead, ReadoutAdapter
from executors.executors import build_model_single_hidden, build_model_two_hidden
from sklearn.svm import SVR
from sklearn.ensemble import BaggingRegressor
# from sklearn.multioutput import MultiOutputSVR # REMOVED (Not in sklearn)

print("--- PART 2: BLIND TEST GENERATION ---")

# --- CUSTOM WRAPPER for Multi-Target SVM ---
class MultiTargetSVRWrapper:
    def __init__(self, estimators):
        self.estimators = estimators
    def fit(self, X, y):
        # y is expected to be (N, n_targets)
        for i, est in enumerate(self.estimators):
            print(f"    Fitting SVM Target {i}...")
            est.fit(X, y[:, i])
    def predict(self, X):
        preds = []
        for est in self.estimators:
            preds.append(est.predict(X))
        return np.column_stack(preds)

# 1. Load Best Configuration
config_file = "best_ensemble_config.json"
if not os.path.exists(config_file):
    raise FileNotFoundError(f"Configuration file {config_file} not found! Run Part 1 first.")

with open(config_file, 'r') as f:
    ensemble_config = json.load(f)

print("Loaded Best Ensemble Configuration:")
print(f"  Best MEE: {ensemble_config['best_mee']:.4f}")
print(f"  Timestamp: {ensemble_config['timestamp']}")

conf = ensemble_config['config']
pt_conf = conf['pytorch']
k_conf = conf['keras']
svm_conf = conf['svm']

# 2. Reload FULL Dataset (Train + Val + Internal Test)
print("Reloading FULL dataset (Train + Val + Internal Test) for Retraining...")
# We set validation_ratio=0.0 and test_ratio=0.0 to get a single tensor containing ALL labeled data.
# The PyTorch ensemble will still use K-Fold on this full dataset to maintain diversity/validation.
train_loader_full, _, _, INPUT_SIZE, OUTPUT_SIZE, target_scaler_full = get_ml_cup_data(
    BATCH_SIZE, 
    validation_ratio=0.0, 
    test_ratio=0.0,       
    scale_target=True,
    num_workers=0
)

X_full = train_loader_full.dataset.X
y_full = train_loader_full.dataset.y
print(f"Full Dataset Shape: {X_full.shape}")

# --- 3. RETRAIN PYTORCH (K-Fold on Full Data) ---
pytorch_models_full = []
if pt_conf['count'] > 0 and pt_conf.get('params'):
    print(f"\nRetraining {pt_conf['count']} PyTorch Models (K-Fold on Full Data)...")
    p_params = pt_conf['params']
    epochs_conf = int(pt_conf.get('epochs', 2000))
    
    # Create new folds on the full dataset for the ensemble
    kf_full = KFold(n_splits=pt_conf['count'], shuffle=True, random_state=42)
    folds_full = list(kf_full.split(X_full))
    
    for idx in range(pt_conf['count']):
        # Get fold indices from the FULL dataset
        train_idx, val_idx = folds_full[idx]
        X_f_train, y_f_train = X_full[train_idx], y_full[train_idx]
        X_f_val, y_f_val = X_full[val_idx], y_full[val_idx]
        
        print(f"  Training PyTorch Full Model {idx+1}/{pt_conf['count']}...")
        
        # Create loader for this fold
        f_dataset = TensorDataset(X_f_train, y_f_train)
        f_loader = DataLoader(f_dataset, batch_size=BATCH_SIZE, shuffle=True)
        
        # Build Model
        hidden_sizes = [int(p_params.get('hidden_size', 32))] * int(p_params.get('n_layers', 2))
        act = p_params.get('activation', 'relu')
        dropout_val = float(p_params.get('dropout', 0.0))
        
        base = StandardFeedForwardNet(INPUT_SIZE, hidden_sizes, OUTPUT_SIZE, act, dropout=dropout_val)
        model = ModelWithHead(base, ReadoutAdapter(OUTPUT_SIZE, OUTPUT_SIZE, 'regression')).to(DEVICE)
        
        opt = torch.optim.SGD(model.parameters(), 
                              lr=float(p_params.get('lr', 0.01)), 
                              momentum=float(p_params.get('momentum', 0.9)), 
                              nesterov=True,
                              weight_decay=float(p_params.get('weight_decay', 0.0)))
        crit = torch.nn.MSELoss()
        
        # Train with Early Stopping
        early_stopper = EarlyStopper(patience=EARLY_STOPPING_PATIENCE, mode='min')
        for epoch in range(epochs_conf):
            model.train()
            for x, y in f_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                if USE_GAUSSIAN_NOISE:
                    noise = torch.randn_like(x) * NOISE_STD
                    x = x + noise
                opt.zero_grad()
                loss = crit(model(x), y)
                loss.backward()
                opt.step()
            
            # Validation Check (Measures MEE on the Unseen Fold)
            model.eval()
            with torch.no_grad():
                x_v = X_f_val.to(DEVICE)
                pred_v = model(x_v).cpu().numpy()
                
                # Unscale targets/preds for real MEE
                y_v_real = target_scaler_full.inverse_transform(y_f_val.numpy())
                pred_v_real = target_scaler_full.inverse_transform(pred_v)
                val_mee = np.mean(np.linalg.norm(y_v_real - pred_v_real, axis=1))
                
                early_stopper(val_mee, model)
                if early_stopper.early_stop:
                    print(f"    Early stopping at epoch {epoch} (Best Val MEE: {early_stopper.best_score:.4f})")
                    break
        
        print(f"    Val MEE: {early_stopper.best_score:.4f}")
                    
        if early_stopper.best_state:
            model.load_state_dict(early_stopper.best_state)
        
        pytorch_models_full.append(model)

# --- 4. RETRAIN KERAS (Full Data) ---
keras_models_full = []
if k_conf['count'] > 0 and k_conf.get('params'):
    print(f"\nRetraining {k_conf['count']} Keras Models...")
    kp = k_conf['params']
    k_meta = {"n_features_in_": INPUT_SIZE, "n_outputs_": OUTPUT_SIZE}
    X_np_full = X_full.numpy()
    y_np_full = y_full.numpy()
    
    for i in range(k_conf['count']):
        print(f"  Training Keras Full Model {i+1}...")
        seed = 42 + i
        
        lr = kp.get('learning_rate', 0.01)
        l1 = kp.get('lambda_1', 1e-4)
        act1 = kp.get('activation_1', 'relu')
        drop1 = kp.get('dropout_1', 0.2)
        l2 = kp.get('lambda_2', None)
        
        if l2:
            u1, u2 = kp.get('unit1', 12), kp.get('unit2', 12)
            act2, drop2 = kp.get('activation_2', 'relu'), kp.get('dropout_2', 0.2)
            model = build_model_two_hidden(k_meta, u1, u2, seed, lr, drop1, drop2, l1, l2, act1, act2)
        else:
            u1 = kp.get('unit1', 12)
            model = build_model_single_hidden(k_meta, u1, seed, lr, drop1, l1, act1)
            
        # Noise
        X_tr_k = X_np_full.copy()
        if USE_GAUSSIAN_NOISE:
            noise = np.random.normal(0, NOISE_STD, X_tr_k.shape)
            X_tr_k += noise
            
        es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)
        model.fit(X_tr_k, y_np_full, epochs=500, verbose=0, batch_size=BATCH_SIZE, validation_split=0.1, callbacks=[es])
        keras_models_full.append(model)

# --- 5. RETRAIN SVM (Full Data) ---
svm_models_full = []
if svm_conf['count'] > 0 and svm_conf.get('params'):
    print(f"\nRetraining {svm_conf['count']} SVM Models...")
    X_np_full = X_full.numpy()
    y_np_full = y_full.numpy()
    sp = svm_conf['params']
    
    estimators_list = []
    for t_idx in range(OUTPUT_SIZE):
        target_key = f"target_{t_idx}"
        s_conf = sp.get(target_key)
        if s_conf:
            base_svr = SVR(kernel='rbf', C=s_conf.get('C', 1.0), epsilon=s_conf.get('epsilon', 0.1), gamma=s_conf.get('gamma', 'scale'))
            regr = BaggingRegressor(estimator=base_svr, n_estimators=svm_conf['count'], 
                                    random_state=42+t_idx, max_samples=0.8, bootstrap=True, n_jobs=-1)
            estimators_list.append(regr)
            
    if len(estimators_list) == OUTPUT_SIZE:
        msvr = MultiTargetSVRWrapper(estimators_list)
        msvr.fit(X_np_full, y_np_full)
        svm_models_full.append(msvr)

# --- 6. UPDATE FULL ENSEMBLE ---
print("\nConstructing Final Ensemble...")
full_ensemble = HeterogeneousEnsemble(pytorch_models_full, keras_models_full, svm_models_full, target_scaler=target_scaler_full)

# IMPORTANT: Update global train_loader for next cell (Blind Test) stats
train_loader = train_loader_full 

print("Done. Ensemble retrained on complete dataset. Ready for Blind Test.")


--- PART 2: BLIND TEST GENERATION ---
Loaded Best Ensemble Configuration:
  Best MEE: 16.7518
  Timestamp: 2026-01-20 18:30:12
Reloading FULL dataset (Train + Val + Internal Test) for Retraining...
Parsing ML-CUP data (TR only)...
Data Split: Train=500, Val=0, Internal Test=0
Applying scaling StandardScaler to Inputs...
Applying StandardScaler to Targets...
Full Dataset Shape: torch.Size([500, 12])

Retraining 5 PyTorch Models (K-Fold on Full Data)...
  Training PyTorch Full Model 1/5...
  Training PyTorch Full Model 2/5...
  Training PyTorch Full Model 3/5...
  Training PyTorch Full Model 4/5...
    Early stopping at epoch 856 (Best Val MEE: 19.6280)
  Training PyTorch Full Model 5/5...
    Early stopping at epoch 1041 (Best Val MEE: 17.6844)

Retraining 5 SVM Models...
    Fitting SVM Target 0...
    Fitting SVM Target 1...
    Fitting SVM Target 2...
    Fitting SVM Target 3...

Constructing Final Ensemble...
Done. Ensemble retrained on complete dataset. Ready for Blind Test.


In [ ]:
# --- BLIND TEST PREDICTION ---
print("Generating Blind Test Output...")

# Load Blind Test Data
blind_df = pd.read_csv("data/MLC25/ML-CUP25-TS.csv", comment='#', header=None)
# Assuming ID is first col? Check template
# Template file: data/MLC25/template-example-with-random-outputs_ML-CUP25-TS.csv
# Usually 1st col ID, rest features.
# Let's inspect shape
print(f"Blind Test Shape: {blind_df.shape}")

# Preprocess
# Remove ID
X_blind = blind_df.iloc[:, 1:].values
ids = blind_df.iloc[:, 0].values

# Scale Input (Must match training data scaling)
# We need the scaler fitted on training data.
# 'train_loader.dataset.transform'? No, dataset stores tensors.
# 'get_ml_cup_data' returns data already scaled if scale_target=True, 
# BUT it scales X and Y independently. X is not returned as a scaler object?
# The loader code handles scaling internally. 
# We need to manually scale X_blind using the statistics of X_train.
X_train_np = train_loader.dataset.X.numpy()
mean = X_train_np.mean(axis=0)
std = X_train_np.std(axis=0) + 1e-8
X_blind_scaled = (X_blind - mean) / std

# Predict
y_blind_pred = full_ensemble.predict(X_blind_scaled)

# Format Output
# Columns: ID, y1, y2, y3, y4 (header included? Check template)
output_df = pd.DataFrame(y_blind_pred, columns=[f"output_{i+1}" for i in range(OUTPUT_SIZE)])
output_df.insert(0, "id", ids)

# Headers logic based on template
# Load template to check header format
# template_df = pd.read_csv("data/MLC25/template-example-with-random-outputs_ML-CUP25-TS.csv")
# For now we write standard CSV without header if template has comments

output_path = "data/MLC25/submission_ensemble.csv"
output_df.to_csv(output_path, index=False)
print(f"Saved submission to {output_path}")
